<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_06_training_dataset_construction/stage_06_training_dataset_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_06_training_dataset_construction**




## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.

2. Generación de ventanas X e y para train, valid y test

    En este paso se generan las ventanas de entrada (X) y los targets (y) para los conjuntos de entrenamiento, validación y prueba.
    Se trabaja con un window_size de 90 minutos y se construyen datasets independientes para cada horizonte de predicción: 30, 60 y 90 minutos.

3. Escalado de ventanas

    En este paso se aplica un proceso de normalización/estandarización a las ventanas generadas, utilizando un scaler entrenado únicamente con el set de entrenamiento para cada horizonte de predicción.
    De esta forma, se aseguran valores comparables entre features y se evita data leakage.
    El scaler ajustado se guarda para poder transformar consistentemente los conjuntos de validación y prueba.

4. Guardado de ventanas escaladas

    En este paso se almacenan en disco las ventanas ya escaladas de train, valid y test, correspondientes a cada horizonte de predicción (30, 60 y 90 minutos).
    Esto permite reutilizar los datasets en etapas posteriores sin necesidad de repetir el preprocesamiento.



## **0. Configuración del Entorno**


### 0.1. Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [ ]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [5]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib
import os
import json
import logging
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

# ----------------------------
# Logging
# ----------------------------
logging.basicConfig(
    level=os.environ.get("LOG_LEVEL", "INFO"),
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("stage_06_training_dataset_construction")

In [82]:
# ============================================================
# Paths / IO (via env o defaults)
# ============================================================
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

IN_PARQUET_TRAIN = Path(os.environ.get("IN_PARQUET_TRAIN", "data/splits/mnq_train.parquet"))
IN_PARQUET_VALID = Path(os.environ.get("IN_PARQUET_VALID", "data/splits/mnq_valid.parquet"))
IN_PARQUET_TEST = Path(os.environ.get("IN_PARQUET_TEST", "data/splits/mnq_test.parquet"))

IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/features_target_summary.json"))

OUT_WINDOWS_60_TRAIN = Path(os.environ.get("OUT_WINDOWS_60", "data/windows/windows_train_60.npz"))
OUT_WINDOWS_60_VALID = Path(os.environ.get("OUT_WINDOWS_60", "data/windows/windows_valid_60.npz"))
OUT_WINDOWS_60_TEST  = Path(os.environ.get("OUT_WINDOWS_60", "data/windows/windows_test_60.npz"))

OUT_WINDOWS_90_TRAIN = Path(os.environ.get("OUT_WINDOWS_90", "data/windows/windows_train_90.npz"))
OUT_WINDOWS_90_VALID = Path(os.environ.get("OUT_WINDOWS_90", "data/windows/windows_valid_90.npz"))
OUT_WINDOWS_90_TEST  = Path(os.environ.get("OUT_WINDOWS_90", "data/windows/windows_test_90.npz"))

# Escalados

OUT_WINDOWS_60_TRAIN_Z = Path(os.environ.get("OUT_WINDOWS_60_Z", "data/windows/scaled/windows_train_60_z.npz"))
OUT_WINDOWS_60_VALID_Z = Path(os.environ.get("OUT_WINDOWS_60_Z", "data/windows/scaled/windows_valid_60_z.npz"))
OUT_WINDOWS_60_TEST_Z  = Path(os.environ.get("OUT_WINDOWS_60_Z", "data/windows/scaled/windows_test_60_z.npz"))

OUT_WINDOWS_90_TRAIN_Z = Path(os.environ.get("OUT_WINDOWS_90_Z", "data/windows/scaled/windows_train_90_z.npz"))
OUT_WINDOWS_90_VALID_Z = Path(os.environ.get("OUT_WINDOWS_90_Z", "data/windows/scaled/windows_valid_90_z.npz"))
OUT_WINDOWS_90_TEST_Z  = Path(os.environ.get("OUT_WINDOWS_90_Z", "data/windows/scaled/windows_test_90_z.npz"))

OUT_SCALER = Path(os.environ.get("OUT_SCALER", "data/windows/scaled/global_scaler.pkl"))

#OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/splits_summary.json"))

#PARA NOTEBOOK

IN_PARQUET_TRAIN = DRIVE_DIR / IN_PARQUET_TRAIN
IN_PARQUET_VALID = DRIVE_DIR / IN_PARQUET_VALID
IN_PARQUET_TEST = DRIVE_DIR / IN_PARQUET_TEST
IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT

OUT_WINDOWS_60_TRAIN = DRIVE_DIR / OUT_WINDOWS_60_TRAIN
OUT_WINDOWS_60_VALID = DRIVE_DIR / OUT_WINDOWS_60_VALID
OUT_WINDOWS_60_TEST = DRIVE_DIR / OUT_WINDOWS_60_TEST

OUT_WINDOWS_90_TRAIN = DRIVE_DIR / OUT_WINDOWS_90_TRAIN
OUT_WINDOWS_90_VALID = DRIVE_DIR / OUT_WINDOWS_90_VALID
OUT_WINDOWS_90_TEST = DRIVE_DIR / OUT_WINDOWS_90_TEST

OUT_WINDOWS_60_TRAIN_Z = DRIVE_DIR / OUT_WINDOWS_60_TRAIN_Z
OUT_WINDOWS_60_VALID_Z = DRIVE_DIR / OUT_WINDOWS_60_VALID_Z
OUT_WINDOWS_60_TEST_Z = DRIVE_DIR / OUT_WINDOWS_60_TEST_Z

OUT_WINDOWS_90_TRAIN_Z = DRIVE_DIR / OUT_WINDOWS_90_TRAIN_Z
OUT_WINDOWS_90_VALID_Z = DRIVE_DIR / OUT_WINDOWS_90_VALID_Z
OUT_WINDOWS_90_TEST_Z = DRIVE_DIR / OUT_WINDOWS_90_TEST_Z

OUT_SCALER = DRIVE_DIR / OUT_SCALER

In [83]:
#En el script de .py esto se debe leer desde el params.yaml
gestation_window_start = '08:20'
gestation_window_end = '08:49'

## **1. Carga de datos**

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [32]:
def load_mnq_parquet(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el parquet de entrada: {path}")

    log.info("[OK] Cargando parquet: %s", path)

    df = pd.read_parquet(path)

    # Asegurar DatetimeIndex
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # Orden temporal explícito
    df = df.sort_index()

    return df


In [33]:
mnq_train = load_mnq_parquet(IN_PARQUET_TRAIN)
mnq_valid = load_mnq_parquet(IN_PARQUET_VALID)
mnq_test = load_mnq_parquet(IN_PARQUET_TEST)

### 1.2. Información de datasets


In [35]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [70]:
info_dataset(mnq_train, 'mnq_train')
info_dataset(mnq_valid, 'mnq_valid')
info_dataset(mnq_test, 'mnq_test')

Información del dataset mnq_train:

	Cantidad de días: 912
	Registros por día: 421
	Hora diaria de inicio 07:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_valid:

	Cantidad de días: 195
	Registros por día: 421
	Hora diaria de inicio 07:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_test:

	Cantidad de días: 196
	Registros por día: 421
	Hora diaria de inicio 07:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York



(196, np.float64(421.0))

### 1.3. Carga de listado de features y targets

In [36]:
import json
#/content/drive/MyDrive/neural_profit/reports/features_target_summary.json
with open(IN_ARTIFACT, "r", encoding="utf-8") as f:
    features_target_summary = json.load(f)

features = features_target_summary["schema"]["features"]
targets = features_target_summary["schema"]["targets"]

print("Features:", features)
print("Targets:", targets)

Features: ['open', 'high', 'low', 'close', 'price_ema60', 'momentum_10', 'roc_30', 'roc_60']
Targets: ['delta_pts_60', 'delta_pts_90']


In [37]:
targets

['delta_pts_60', 'delta_pts_90']

## **2. Carga de features para cada horizonte 60 y 90min**

Definimos el target de cada horizonte:

In [38]:
target_60 = targets[0]
target_90 = targets[1]

Luego definimos el listado de features para cada horizonte:

In [16]:
features

['open',
 'high',
 'low',
 'close',
 'price_ema60',
 'momentum_10',
 'roc_30',
 'roc_60']

In [17]:
features_60 = [f for f in features if f != "roc_30"]
features_90 = [f for f in features if f != "roc_60"]

In [18]:
features_60

['open', 'high', 'low', 'close', 'price_ema60', 'momentum_10', 'roc_60']

In [19]:
features_90

['open', 'high', 'low', 'close', 'price_ema60', 'momentum_10', 'roc_30']

## **3. Filtrado de datasets ajustado a la ventana de gestación**

Anteriormente habíamos mencionado que entrenariamos el modelo con los datos de la ventana de gestación.

In [44]:
gestation_window_start, gestation_window_end

('08:20', '08:49')

En este punto, es necesario filtrar los dataset dentro de esos horarios:


In [53]:
def filter_gestation_window(
    df: pd.DataFrame,
    start: str = gestation_window_start,
    end: str = gestation_window_end,
    date_col: str = "date",
) -> pd.DataFrame:
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El DataFrame debe tener un DatetimeIndex en df.index.")

    if date_col not in df.columns:
        raise KeyError(f"No se encontró la columna '{date_col}' en el DataFrame.")

    # 1) Orden temporal antes de filtrar
    df = df.sort_index()

    # 2) Consistencia: date_col debe coincidir con la fecha del índice
    # (esto previene 'mezclar días' por datos corruptos/desalineados)
    idx_dates = df.index.date
    col_dates = pd.to_datetime(df[date_col]).dt.date
    if not (col_dates.values == idx_dates).all():
        raise ValueError(
            f"Inconsistencia entre '{date_col}' y la fecha del DatetimeIndex. "
            "Revise que no haya registros mal asignados a otro día."
        )

    # 3) Filtrado por ventana horaria (no cruza días; opera dentro de cada fecha)
    out = df.between_time(start, end, inclusive="both")

    # 4) Orden temporal después de filtrar
    out = out.sort_index()

    # 5) Validación adicional: por cada 'date', el índice debe pertenecer a ese mismo día
    out_idx_dates = out.index.date
    out_col_dates = pd.to_datetime(out[date_col]).dt.date
    if not (out_col_dates.values == out_idx_dates).all():
        raise ValueError(
            f"El filtrado generó inconsistencia entre '{date_col}' y el índice. "
            "Esto no debería ocurrir; revise el dataset."
        )

    return out


In [54]:
mnq_train_g = filter_gestation_window(mnq_train)
mnq_valid_g = filter_gestation_window(mnq_valid)
mnq_test_g  = filter_gestation_window(mnq_test)

In [71]:
n_days_train, median_by_day_train = info_dataset(mnq_train_g, 'mnq_train_g')
n_days_valid, median_by_day_valid = info_dataset(mnq_valid_g, 'mnq_valid_g')
n_days_test, median_by_day_test = info_dataset(mnq_test_g, 'mnq_test_g')

Información del dataset mnq_train_g:

	Cantidad de días: 912
	Registros por día: 30
	Hora diaria de inicio 08:20
	Hora diaria de final 08:49
	Zona horaria: America/New_York

Información del dataset mnq_valid_g:

	Cantidad de días: 195
	Registros por día: 30
	Hora diaria de inicio 08:20
	Hora diaria de final 08:49
	Zona horaria: America/New_York

Información del dataset mnq_test_g:

	Cantidad de días: 196
	Registros por día: 30
	Hora diaria de inicio 08:20
	Hora diaria de final 08:49
	Zona horaria: America/New_York



In [72]:
n_days_train, median_by_day_train

(912, np.float64(30.0))

## **4. Definición de windows size**

El tamaño de la ventana de entrada se fija en la cantidad de  registros por día, porque corresponde a la ventana de gestación del movimiento, es decir, el intervalo intradía en el que el mercado concentra la información relevante previa a la expansión del precio.

Al filtrar previamente los datos por esta franja horaria y construir una única secuencia por día, cada muestra representa un contexto temporal homogéneo, evita mezclar dinámicas de distintos momentos de la sesión y alinea la longitud de la secuencia con la lógica operativa del problema, no con el horizonte del target ni con el período de cálculo de los indicadores técnicos.


In [73]:
if median_by_day_train == median_by_day_valid == median_by_day_test:
    window_size = int(median_by_day_train)
    print(f"window_size: {window_size}")
else:
    raise ValueError(
        f"Window size mismatch: "
        f"train={median_by_day_train}, "
        f"valid={median_by_day_valid}, "
        f"test={median_by_day_test}"
    )

window_size: 30


Las rutas para las ventanas son:

In [74]:
OUT_WINDOWS_60_TRAIN

PosixPath('/content/drive/MyDrive/neural_profit/data/windows/windows_train_60.npz')

In [60]:
OUT_WINDOWS_90

PosixPath('/content/drive/MyDrive/neural_profit/data/train/windows_90.npz')

## 5. **Generación de ventanas**

Los datasets `mnq_train_g`, `mnq_valid_g` y `mnq_test_g` se encuentran preagrupados por día, de modo que cada día corresponde exactamente a una única ventana temporal fija, sin generación de ventanas deslizantes dentro de la jornada.

- Agrupamiento diario:

  Cada muestra del dataset corresponde a un día de operación.
  Cada día contiene exactamente 30 registros consecutivos minuto a minuto, comprendidos entre las 08:20 y las 08:49 (America/New_York).

- Estructura de las features:

  Para cada día, los 30 registros forman una matriz de features de dimensión 30x7, donde cada fila representa un minuto y cada columna una variable de entrada.

- Definición del target:

  El target ya está definido a nivel de cada fila del dataset y representa el delta de puntos hacia adelante (por ejemplo, a 60 o 90 minutos).

  Para cada día, el objetivo del modelo es predecir el bloque completo de 30 valores de target, alineados uno a uno con los 30 registros de entrada.

  De este modo, cada muestra se define como:

  Entrada (30x8) ⟶ Salida (1x30)

  sin superposición entre días, sin ventanas internas deslizantes y preservando estrictamente la coherencia temporal.


### **5.1. Funciones para construcción de ventanas secuencia a secuencia**

#### **5.1.1. Construcción de ventanas**

In [ ]:
import numpy as np

def build_daily_seq2seq_windows(
    df,
    features,
    target_col,
    window_size: int,
    date_col: str = "date",
):
    """
    Construye muestras alineadas con un esquema SEQ2SEQ diario:

        Entrada  X: (window_size, n_features)  -> 30 x 8
        Salida   y: (window_size,)             -> 1 x 30

    Supuestos clave (alineados con el pipeline actual):
    - El dataset ya viene preagrupado por día.
    - Cada día contiene exactamente `window_size` registros consecutivos
      (por ejemplo, 30 minutos entre 08:20 y 08:49).
    - NO se generan ventanas deslizantes dentro del día.
    - Se obtiene UNA muestra por día.

    Parámetros:
    - df: DataFrame que contiene columnas [date_col] + features + target_col
    - features: lista de columnas de entrada
      (por ejemplo: features_60 o features_90)
    - target_col: columna objetivo
      (por ejemplo: delta_pts_60 o delta_pts_90)
    - window_size: cantidad de registros por día (default: 30)
    - date_col: columna usada para agrupar por día (default: "date")

    Retorna:
    - X: np.ndarray con shape (n_dias_validos, window_size, n_features)
    - y: np.ndarray con shape (n_dias_validos, window_size)
    """
    X, y = [], []

    # 1) Agrupamiento diario: cada grupo representa una ventana fija del día
    for _, grupo in df.groupby(date_col):
        # Asegurar orden temporal dentro del día
        grupo = grupo.sort_index()

        # 2) Validar que el día tenga exactamente window_size registros
        if len(grupo) != window_size:
            continue

        # 3) Extraer la matriz de features (30 x n_features)
        X_day = grupo[features].values

        # 4) Extraer el bloque completo de targets (30,)
        y_day = grupo[target_col].values

        # 5) Chequeo de NaN para evitar muestras inválidas
        if np.isnan(X_day).any() or np.isnan(y_day).any():
            continue

        X.append(X_day)
        y.append(y_day)

    return np.array(X), np.array(y)

#### **5.1.2. Construcción de ventanas para train, valid y test**

In [75]:
import os
import numpy as np
from pathlib import Path


def prepare_or_load_seq2seq_windows(
    mnq_train,
    mnq_valid,
    mnq_test,
    features,
    target_col: str,
    window_size: int,
    out_windows_train,
    out_windows_valid,
    out_windows_test,
    date_col: str = "date",
):
    """
    Genera (si no existe) o carga (si ya existe) las ventanas SEQ2SEQ diarias
    usando `build_daily_seq2seq_windows`, y las guarda como .npz comprimido.

    Cada muestra:
      X: (window_size, n_features)
      y: (window_size,)

    Nota: out_windows_* puede ser str o pathlib.Path (por ejemplo PosixPath).
    """

    def _to_path(p) -> Path:
        return p if isinstance(p, Path) else Path(str(p))

    def _ensure_parent_dir(path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)

    def _load_or_build(split_name: str, df, out_path):
        out_path = _to_path(out_path)
        _ensure_parent_dir(out_path)

        if not out_path.exists():
            print(f"No existe -> Generando {split_name} para '{target_col}' y guardando en: {out_path}")
            X, y = build_daily_seq2seq_windows(
                df=df,
                features=features,
                target_col=target_col,
                window_size=window_size,
                date_col=date_col,
            )
            np.savez_compressed(out_path, X=X, y=y)
            print(f"Guardado: {out_path} | X: {X.shape} | y: {y.shape}")
            return X, y

        print(f"Ya existe -> Cargando {split_name} desde: {out_path}")
        data = np.load(out_path)
        X, y = data["X"], data["y"]
        print(f"Cargado: {out_path} | X: {X.shape} | y: {y.shape}")
        return X, y

    X_train, y_train = _load_or_build("train", mnq_train, out_windows_train)
    X_valid, y_valid = _load_or_build("valid", mnq_valid, out_windows_valid)
    X_test,  y_test  = _load_or_build("test",  mnq_test,  out_windows_test)

    return X_train, y_train, X_valid, y_valid, X_test, y_test


#### **5.1.3. Función para revisar composición de ventanas**

In [78]:
import numpy as np

def xy_info_seq2seq(
    horizon_min: int,
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    n_features: int | None = None,
):
    """
    Imprime información resumida de los arrays X/y para el esquema diario SEQ2SEQ.

    Esperado (nuestro proyecto):
      X: (n_samples, window_size, n_features)   -> por ejemplo (912, 30, 8)
      y: (n_samples, window_size)              -> por ejemplo (912, 30)

    Parámetros:
    - horizon_min: horizonte del target (60 o 90), solo para rotular
    - n_features: opcional; si se pasa, valida que X.shape[2] coincida
    """

    print(f"Información X/y para horizonte {horizon_min} min:")

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación",    X_valid, y_valid),
        ("testeo",        X_test,  y_test),
    ]:
        print(f"\nSet de {name}:")

        # --- Formas ---
        print(f"\tX shape: {X.shape}")
        print(f"\ty shape: {y.shape}")

        # --- Validaciones básicas ---
        if X.ndim != 3:
            print("\t[AVISO] X no es 3D. Se esperaba (n_samples, window_size, n_features).")
        else:
            n_samples, window_size, n_feat = X.shape
            print(f"\t{n_samples} días/muestras (n_samples).")
            print(f"\t{window_size} pasos temporales por día (window_size).")
            print(f"\t{n_feat} features por paso.")

            if n_features is not None and n_feat != n_features:
                print(f"\t[AVISO] n_features esperado={n_features}, encontrado={n_feat}.")

        if y.ndim != 2:
            print("\t[AVISO] y no es 2D. Se esperaba (n_samples, window_size).")
        else:
            if X.shape[0] != y.shape[0]:
                print(f"\t[AVISO] n_samples difiere: X={X.shape[0]} vs y={y.shape[0]}.")
            if X.ndim == 3 and X.shape[1] != y.shape[1]:
                print(f"\t[AVISO] window_size difiere: X={X.shape[1]} vs y={y.shape[1]}.")

        # --- Estadísticas de y (sobre TODOS los valores del bloque) ---
        y_flat = np.asarray(y).ravel()
        print(
            "\tDistribución y (flatten): "
            f"mean={y_flat.mean():.6f}, std={y_flat.std():.6f}, "
            f"min={y_flat.min():.6f}, max={y_flat.max():.6f}"
        )


### **5.2. Generación de ventanas (H = 60min)**

In [77]:
X_train_60, y_train_60, X_valid_60, y_valid_60, X_test_60, y_test_60 = prepare_or_load_seq2seq_windows(
    mnq_train=mnq_train_g,
    mnq_valid=mnq_valid_g,
    mnq_test=mnq_test_g,
    features=features_60,
    target_col="delta_pts_60",
    window_size=window_size,
    out_windows_train=OUT_WINDOWS_60_TRAIN,
    out_windows_valid=OUT_WINDOWS_60_VALID,
    out_windows_test=OUT_WINDOWS_60_TEST,
    date_col="date",
)

Ya existe -> Cargando train desde: /content/drive/MyDrive/neural_profit/data/windows/windows_train_60.npz
Cargado: /content/drive/MyDrive/neural_profit/data/windows/windows_train_60.npz | X: (912, 30, 7) | y: (912, 30)
Ya existe -> Cargando valid desde: /content/drive/MyDrive/neural_profit/data/windows/windows_valid_60.npz
Cargado: /content/drive/MyDrive/neural_profit/data/windows/windows_valid_60.npz | X: (195, 30, 7) | y: (195, 30)
Ya existe -> Cargando test desde: /content/drive/MyDrive/neural_profit/data/windows/windows_test_60.npz
Cargado: /content/drive/MyDrive/neural_profit/data/windows/windows_test_60.npz | X: (196, 30, 7) | y: (196, 30)


In [79]:
xy_info_seq2seq(
    horizon_min=60,
    X_train=X_train_60, y_train=y_train_60,
    X_valid=X_valid_60, y_valid=y_valid_60,
    X_test=X_test_60,   y_test=y_test_60,
    n_features=len(features_60),
)

Información X/y para horizonte 60 min:

Set de entrenamiento:
	X shape: (912, 30, 7)
	y shape: (912, 30)
	912 días/muestras (n_samples).
	30 pasos temporales por día (window_size).
	7 features por paso.
	Distribución y (flatten): mean=-0.126124, std=53.237769, min=-484.000000, max=475.000000

Set de validación:
	X shape: (195, 30, 7)
	y shape: (195, 30)
	195 días/muestras (n_samples).
	30 pasos temporales por día (window_size).
	7 features por paso.
	Distribución y (flatten): mean=0.552350, std=50.311440, min=-284.250000, max=256.500000

Set de testeo:
	X shape: (196, 30, 7)
	y shape: (196, 30)
	196 días/muestras (n_samples).
	30 pasos temporales por día (window_size).
	7 features por paso.
	Distribución y (flatten): mean=-0.912202, std=73.215870, min=-404.000000, max=542.500000


### **5.3. Generación de ventanas (H = 90min)**

In [80]:
X_train_90, y_train_90, X_valid_90, y_valid_90, X_test_90, y_test_90 = prepare_or_load_seq2seq_windows(
    mnq_train=mnq_train_g,
    mnq_valid=mnq_valid_g,
    mnq_test=mnq_test_g,
    features=features_90,
    target_col="delta_pts_90",
    window_size=window_size,
    out_windows_train=OUT_WINDOWS_90_TRAIN,
    out_windows_valid=OUT_WINDOWS_90_VALID,
    out_windows_test=OUT_WINDOWS_90_TEST,
    date_col="date",
)

No existe -> Generando train para 'delta_pts_90' y guardando en: /content/drive/MyDrive/neural_profit/data/windows/windows_train_90.npz
Guardado: /content/drive/MyDrive/neural_profit/data/windows/windows_train_90.npz | X: (912, 30, 7) | y: (912, 30)
No existe -> Generando valid para 'delta_pts_90' y guardando en: /content/drive/MyDrive/neural_profit/data/windows/windows_valid_90.npz
Guardado: /content/drive/MyDrive/neural_profit/data/windows/windows_valid_90.npz | X: (195, 30, 7) | y: (195, 30)
No existe -> Generando test para 'delta_pts_90' y guardando en: /content/drive/MyDrive/neural_profit/data/windows/windows_test_90.npz
Guardado: /content/drive/MyDrive/neural_profit/data/windows/windows_test_90.npz | X: (196, 30, 7) | y: (196, 30)


In [81]:
xy_info_seq2seq(
    horizon_min=90,
    X_train=X_train_90, y_train=y_train_90,
    X_valid=X_valid_90, y_valid=y_valid_90,
    X_test=X_test_90,   y_test=y_test_90,
    n_features=len(features_90),
)

Información X/y para horizonte 90 min:

Set de entrenamiento:
	X shape: (912, 30, 7)
	y shape: (912, 30)
	912 días/muestras (n_samples).
	30 pasos temporales por día (window_size).
	7 features por paso.
	Distribución y (flatten): mean=0.413889, std=78.674101, min=-537.000000, max=570.000000

Set de validación:
	X shape: (195, 30, 7)
	y shape: (195, 30)
	195 días/muestras (n_samples).
	30 pasos temporales por día (window_size).
	7 features por paso.
	Distribución y (flatten): mean=-0.161496, std=71.009719, min=-258.750000, max=295.250000

Set de testeo:
	X shape: (196, 30, 7)
	y shape: (196, 30)
	196 días/muestras (n_samples).
	30 pasos temporales por día (window_size).
	7 features por paso.
	Distribución y (flatten): mean=-1.426148, std=120.970425, min=-422.750000, max=1301.250000


## 3. Escalado de ventanas

En este punto se escalan las ventanas de entrada para que todas las features tengan la misma magnitud, entrenando el scaler con los datos de entrenamiento y aplicándolo luego a validación y test.

### 3.0. Funciones


#### Función para elegir escalador

In [ ]:
def _choose_scaler(scaler_type="standard"):
    st = scaler_type.lower()
    if st in ["standard", "z", "zscore"]:
        return StandardScaler()
    elif st in ["minmax", "min_max"]:
        return MinMaxScaler()
    else:
        raise ValueError("scaler_type debe ser 'standard' o 'minmax'")

#### Función para entrenar un scaler en datos secuenciales 3D (ventanas), tratándolos como una sola tabla 2D de features.

In [ ]:
# -------------------------------------------------------------------------
# _fit_on_3d
#
# Entrada: X_train_3d con forma (n, W, F)
#   n = número de muestras (ventanas)
#   W = lookback (número de pasos en cada ventana)
#   F = número de features por paso
#
# Qué hace:
# - Aplana las dos primeras dimensiones (n, W) → queda una matriz de (n*W, F).
# - Convierte todas las secuencias en un dataset tabular de features.
# - Ajusta el scaler (ej. StandardScaler) sobre todos los valores de todas
#   las ventanas y pasos, feature por feature.
#
# Resultado: devuelve un scaler entrenado con la estadística global de cada
# feature (media, std, min, max, según el tipo de scaler utilizado).
# -------------------------------------------------------------------------

def _fit_on_3d(X_train_3d, scaler):
    n, W, F = X_train_3d.shape
    scaler.fit(X_train_3d.reshape(-1, F))
    return scaler

#### Función para aplicar el scaler de _fit_on_3d y devolver los datos escalados, manteniendo la estructura original (n, W, F).

In [ ]:
# -------------------------------------------------------------------------
# _transform_3d
#
# Entrada: X_3d con forma (n, W, F)
#   n = número de muestras (ventanas)
#   W = lookback (número de pasos en cada ventana)
#   F = número de features por paso
#
# Qué hace:
# - Aplana las dos primeras dimensiones (n, W) → queda una matriz de (n*W, F).
# - Aplica la transformación del scaler entrenado (ej. StandardScaler).
# - Restaura la forma original (n, W, F) para conservar la estructura 3D
#   necesaria en modelos secuenciales (RNN, LSTM, Transformers).
#
# Resultado: devuelve el mismo dataset 3D pero con todos los features escalados
# de manera consistente en cada ventana y paso de tiempo.
# -------------------------------------------------------------------------

def _transform_3d(X_3d, scaler):
    n, W, F = X_3d.shape
    Xf = X_3d.reshape(-1, F)
    Xs = scaler.transform(Xf).reshape(n, W, F)
    return Xs

#### Función para escalar y guardar escalador

In [ ]:
def scale_and_save(
    X_train,
    X_valid=None,
    X_test=None,
    scaler_type="standard",
    scaler_path=f"{drive_path}/3_dataset_preparation/global_scaler.pkl",
    window_size=None,   # si X_* están en 2D (n_samples, window_size*len(features_{target})), pasá window_size para escalar por feature
    verbose=True
):
    """
    Escala X_train (y opcionalmente valid/test) y guarda el escalador.
    - Si X_* es 3D: (n, W, F) -> fit por feature.
    - Si X_* es 2D: (n, W*F). Si pasás window_size=W, reescala por feature reconstruyendo 3D; si no, escala columnas tal cual.

    Return:
        X_train_scaled, X_valid_scaled (o None), X_test_scaled (o None), scaler
    """
    scaler = _choose_scaler(scaler_type)

    # Detectar dimensiones y preparar para fit / transform
    if X_train.ndim == 3:
        # 3D directo
        scaler = _fit_on_3d(X_train, scaler)
        X_train_s = _transform_3d(X_train, scaler)
        X_valid_s = _transform_3d(X_valid, scaler) if X_valid is not None else None
        X_test_s  = _transform_3d(X_test,  scaler) if X_test  is not None else None

    elif X_train.ndim == 2:
        n, tot = X_train.shape
        if window_size is not None:
            # Reescalar por feature: reconstruyo 3D -> escalo -> vuelvo a 2D
            assert tot % window_size == 0, "total de columnas no divisible por window_size"
            F = tot // window_size

            def to3d(X2d):
                return X2d.reshape(X2d.shape[0], window_size, F)

            Xtr3 = to3d(X_train)
            scaler = _fit_on_3d(Xtr3, scaler)

            X_train_s = _transform_3d(Xtr3, scaler).reshape(n, tot)

            if X_valid is not None:
                Xva3 = to3d(X_valid)
                X_valid_s = _transform_3d(Xva3, scaler).reshape(X_valid.shape[0], tot)
            else:
                X_valid_s = None

            if X_test is not None:
                Xte3 = to3d(X_test)
                X_test_s = _transform_3d(Xte3, scaler).reshape(X_test.shape[0], tot)
            else:
                X_test_s = None
        else:
            # Escalado columna a columna (no reconstruye 3D)
            scaler.fit(X_train)
            X_train_s = scaler.transform(X_train)
            X_valid_s = scaler.transform(X_valid) if X_valid is not None else None
            X_test_s  = scaler.transform(X_test)  if X_test  is not None else None
    else:
        raise ValueError("X_train debe ser 2D o 3D.")

    # Guardar escalador
    joblib.dump(scaler, scaler_path)
    if verbose:
        print(f"✅ Scaler guardado en: {scaler_path}")
        print("Shapes escaladas:",
              "X_train", X_train_s.shape,
              "| X_valid", None if X_valid is None else X_valid_s.shape,
              "| X_test",  None if X_test  is None  else X_test_s.shape)

    return X_train_s, X_valid_s, X_test_s, scaler

### 3.1. Escalado para horizonte de 30 minutos (`target_return_30`)

In [ ]:
X_train_30_s, X_valid_30_s, X_test_30_s, scaler_30 = scale_and_save(
    X_train_30, X_valid_30, X_test_30,
    scaler_type="standard",
    scaler_path=f"{drive_path}/3_dataset_preparation/global_scaler_30.pkl",
    window_size=90,   # importante para reescalar por feature
    verbose=True
)

✅ Scaler guardado en: /content/drive/MyDrive/neural_profit/3_dataset_preparation/global_scaler_30.pkl
Shapes escaladas: X_train (193487, 900) | X_valid (41567, 900) | X_test (41567, 900)


### 3.2. Escalado para horizonte de 60 minutos (`target_return_60`)

In [ ]:
X_train_60_s, X_valid_60_s, X_test_60_s, scaler_60 = scale_and_save(
    X_train_60, X_valid_60, X_test_60,
    scaler_type="standard",
    scaler_path=f"{drive_path}/3_dataset_preparation/global_scaler_60.pkl",
    window_size=90,   # importante para reescalar por feature
    verbose=True
)

✅ Scaler guardado en: /content/drive/MyDrive/neural_profit/3_dataset_preparation/global_scaler_60.pkl
Shapes escaladas: X_train (193487, 1080) | X_valid (41567, 1080) | X_test (41567, 1080)


### 3.3. Escalado para horizonte de 90 minutos (`target_return_90`)

In [ ]:
X_train_90_s, X_valid_90_s, X_test_90_s, scaler_90 = scale_and_save(
    X_train_90, X_valid_90, X_test_90,
    scaler_type="standard",
    scaler_path=f"{drive_path}/3_dataset_preparation/global_scaler_90.pkl",
    window_size=90,   # importante para reescalar por feature
    verbose=True
)

✅ Scaler guardado en: /content/drive/MyDrive/neural_profit/3_dataset_preparation/global_scaler_90.pkl
Shapes escaladas: X_train (193487, 1080) | X_valid (41567, 1080) | X_test (41567, 1080)


## 4. Guardamos las ventanas escaladas


Se guardan los conjuntos de ventanas escaladas para cada horizonte, listos para ser usados en el entrenamiento de modelos.

In [ ]:
def xy_scaled_path (target: str):
  path_xy_train = f"{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz"
  path_xy_valid = f"{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz"
  path_xy_test =  f"{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz"
  return path_xy_train, path_xy_valid, path_xy_test

In [ ]:
path_xy_train_30_scaled, path_xy_valid_30_scaled, path_xy_test_30_scaled = xy_scaled_path('30')
path_xy_train_60_scaled, path_xy_valid_60_scaled, path_xy_test_60_scaled = xy_scaled_path('60')
path_xy_train_90_scaled, path_xy_valid_90_scaled, path_xy_test_90_scaled = xy_scaled_path('90')

In [ ]:
def save_or_load_scaled_data(path_train, path_valid, path_test, X_train_s, X_valid_s, X_test_s, y_train, y_valid, y_test):

  if not os.path.exists(path_train):
      print('El archivo no existe -> Guardando X_train_s')
      np.savez_compressed(path_train, X=X_train_s, y=y_train)
      print("Guardado:", path_train)
  else:
      print("Ya existe -> Cargando desde disco:", path_train)
      data_train_s = np.load(path_train)
      X_train_s, y_train = data_train_s["X"], data_train_s["y"]

  if not os.path.exists(path_valid):
      print('El archivo no existe -> Guardando X_valid_s')
      np.savez_compressed(path_valid, X=X_valid_s, y=y_valid)
      print("Guardado:", path_valid)
  else:
      print("Ya existe -> Cargando desde disco:", path_valid)
      data_valid_s = np.load(path_valid)
      X_valid_s, y_valid = data_valid_s["X"], data_valid_s["y"]

  if not os.path.exists(path_test):
      print('El archivo no existe -> Guardando X_test_s')
      np.savez_compressed(path_test, X=X_test_s, y=y_test)
      print("Guardado:", path_test)
  else:
      print("Ya existe -> Cargando desde disco:", path_test)
      data_test_s = np.load(path_test)
      X_test_s, y_test = data_test_s["X"], data_test_s["y"]


In [ ]:
save_or_load_scaled_data(path_xy_train_30_scaled, path_xy_valid_30_scaled, path_xy_test_30_scaled, X_train_30_s, X_valid_30_s, X_test_30_s, y_train_30,  y_valid_30,  y_test_30 )

El archivo no existe -> Guardando X_train_s
Guardado: /content/drive/MyDrive/neural_profit/3_dataset_preparation/xy_windows_scaled/xy_train_30_scaled.npz
El archivo no existe -> Guardando X_valid_s
Guardado: /content/drive/MyDrive/neural_profit/3_dataset_preparation/xy_windows_scaled/xy_valid_30_scaled.npz
El archivo no existe -> Guardando X_test_s
Guardado: /content/drive/MyDrive/neural_profit/3_dataset_preparation/xy_windows_scaled/xy_test_30_scaled.npz


In [ ]:
save_or_load_scaled_data(path_xy_train_60_scaled, path_xy_valid_60_scaled, path_xy_test_60_scaled, X_train_60_s, X_valid_60_s, X_test_60_s, y_train_60,  y_valid_60,  y_test_60 )

El archivo no existe -> Guardando X_train_s
Guardado: /content/drive/MyDrive/neural_profit/3_dataset_preparation/xy_windows_scaled/xy_train_60_scaled.npz
El archivo no existe -> Guardando X_valid_s
Guardado: /content/drive/MyDrive/neural_profit/3_dataset_preparation/xy_windows_scaled/xy_valid_60_scaled.npz
El archivo no existe -> Guardando X_test_s
Guardado: /content/drive/MyDrive/neural_profit/3_dataset_preparation/xy_windows_scaled/xy_test_60_scaled.npz


In [ ]:
save_or_load_scaled_data(path_xy_train_90_scaled, path_xy_valid_90_scaled, path_xy_test_90_scaled, X_train_90_s, X_valid_90_s, X_test_90_s, y_train_90,  y_valid_90,  y_test_90 )

El archivo no existe -> Guardando X_train_s
Guardado: /content/drive/MyDrive/neural_profit/3_dataset_preparation/xy_windows_scaled/xy_train_90_scaled.npz
El archivo no existe -> Guardando X_valid_s
Guardado: /content/drive/MyDrive/neural_profit/3_dataset_preparation/xy_windows_scaled/xy_valid_90_scaled.npz
El archivo no existe -> Guardando X_test_s
Guardado: /content/drive/MyDrive/neural_profit/3_dataset_preparation/xy_windows_scaled/xy_test_90_scaled.npz
